# Day 77: Capstone Part 2 - Modeling & Evaluation

## Reload the Data
Notebooks are often run independently, so we start fresh by reloading the Wine dataset rather than assuming Part 1's variables are still in memory - this is also good practice since a real modeling pipeline should be able to run standalone. Beyond just reloading, this cell does the essential data preparation steps every supervised learning project needs: splitting features (`X`) from the target (`y`), splitting into train/test sets so we can measure performance on unseen data, and scaling the features. `stratify=y` in `train_test_split` ensures the train and test sets each preserve the same class proportions as the full dataset (important since we saw in Part 1 that classes aren't perfectly balanced), and `StandardScaler` rescales every feature to zero mean/unit variance - fit only on the training data (`fit_transform`) and then applied to the test data using those same training statistics (`transform`, not `fit_transform`, to avoid leaking test data information into the scaling).</cell id="cell-1">


In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_wine(as_frame=True)
df = data.frame
X = df.drop(columns=['target'])  # all feature columns
y = df['target']  # the class label to predict

# stratify=y keeps the same class proportions in both train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn scaling params from training data only
X_test_scaled = scaler.transform(X_test)  # apply the SAME scaling to test data (no fitting) to avoid data leakage

## Train Multiple Models
No single algorithm is best for every dataset, so a sound modeling workflow trains and compares several candidates before committing to one - here we try Logistic Regression (a simple, interpretable linear model), SVM (Support Vector Machine, good at finding complex decision boundaries), and Random Forest (an ensemble of decision trees, often strong on tabular data with little tuning). Each model is trained on the same scaled training data and scored with `.score()`, which for classifiers returns accuracy (fraction of correct predictions) on the given data - here the held-out test set. This quick comparison isn't the final word (it uses a single train/test split and default hyperparameters), but it's a fast, standard way to narrow down which algorithm is worth investing further tuning effort into.</cell id="cell-3">


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=200),  # max_iter raised so the solver has room to converge
    'SVM': SVC(),
    'Random Forest': RandomForestClassifier(random_state=42)
}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)  # train on scaled training data
    print(name, 'accuracy:', model.score(X_test_scaled, y_test))  # accuracy on held-out test data

## Tune the Best Model
Default hyperparameters rarely give the best possible performance, so once you've picked a promising algorithm (Random Forest here), the next step is hyperparameter tuning - systematically searching over different settings to find the combination that performs best. `GridSearchCV` exhaustively tries every combination in `param_grid` (here, 3 values of `n_estimators` x 3 values of `max_depth` = 9 combinations), and for each one runs 5-fold cross-validation (`cv=5`): the training data is split into 5 parts, the model is trained on 4 and validated on the 1 remaining, repeated 5 times so every part gets used for validation once, and the scores are averaged. This is more robust than a single train/test split because it reduces the chance that a good (or bad) score was just luck from one particular split. After the search, `grid.best_params_` shows the winning hyperparameter combination, and `grid.best_estimator_` is a model already refit on the full training set using those best parameters, ready to evaluate on the untouched test set.</cell id="cell-5">


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, None]}  # 'None' means trees grow until leaves are pure
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)  # 5-fold cross-validation over every combination
grid.fit(X_train_scaled, y_train)
print('Best params:', grid.best_params_)  # hyperparameter combination with the best average CV score
print('Test accuracy:', grid.best_estimator_.score(X_test_scaled, y_test))  # best model evaluated on the untouched test set

## Final Evaluation Report
A single accuracy number can hide important weaknesses, especially with multiple classes or class imbalance - a model could have decent overall accuracy while performing poorly on one specific class. `classification_report` breaks performance down per class with precision (of the samples predicted as this class, what fraction actually were), recall (of the samples that actually are this class, what fraction did the model correctly find), and F1-score (the harmonic mean of precision and recall, a single number balancing both). This per-class view is what you'd actually want to show a stakeholder or include in a project writeup, since it tells the full story of where the final model does well and where it might still make costly mistakes - much more informative than a single accuracy figure.</cell id="cell-7">


In [ ]:
from sklearn.metrics import classification_report

final_model = grid.best_estimator_  # the tuned Random Forest from the grid search
preds = final_model.predict(X_test_scaled)
print(classification_report(y_test, preds, target_names=data.target_names))  # per-class precision/recall/F1